# Deep Learning - 2024 / 2025

> Adicionar blockquote



Grupo 42 - PMMA

# Dependencies

In [2]:
#! pip install tensorflow-datasets

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

import tensorflow_datasets as tfds # to import the dataset

# Auxiliary functions /code

In [4]:
def count_n_images_iterator(this_iterator, name, what='images'):
    """
    Function to count the number of items (images/batches) in the dataset that
    is an iterator. Does not return. Prints a string.

    Input:
        this_iterator: tf.data.Dataset
            Iterator containing the data. TensorFlow Dataset.
        name: str
            Name of the dataset. Should be 'train','test','validation'
        what: str
            Specifies if we are dealing with single images or batches. Should
            be 'images' or 'batches'.

    Output:
        None
    """

    count_items = 0
    for item, label in this_iterator:
        count_items += 1

    print(f'The {name} dataset has {count_items} {what}.')

In [5]:
def plot_acuracy_loss(history):
    """
    Adapted from: https://www.tensorflow.org/tutorials/images/classification
    """
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']

    loss = history.history['loss']
    val_loss = history.history['val_loss']

    epochs_range = range(epochs)

    plt.figure(figsize=(8, 8))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')
    plt.show()

# Load Data

In [6]:
split0, split1, split2 = tfds.even_splits('train', n=3)

In [7]:
ds_train, ds_info = tfds.load(
    'cats_vs_dogs',
    split=split0,
    shuffle_files=False,
    as_supervised=True,
    with_info=True,
    )

ModuleNotFoundError: No module named 'importlib_resources'

In [ ]:
ds_test, __ = tfds.load(
    'cats_vs_dogs',
    split=split1,
    shuffle_files=False,
    as_supervised=True,
    with_info=True,
    )

In [ ]:
ds_val, __ = tfds.load(
    'cats_vs_dogs',
    split=split2,
    shuffle_files=False,
    as_supervised=True,
    with_info=True,
    )

## Data information

In [ ]:
print(ds_info)

## Explore the data

In [ ]:
fig = tfds.show_examples(ds_train, ds_info, rows=4, cols=4)

In [ ]:
count_n_images_iterator(ds_train, 'train'),
count_n_images_iterator(ds_test, 'test'),
count_n_images_iterator(ds_val, 'validation')

The dataset contains images with diferent shapes, diferent widths and heights. Therefore, they need to be all converted to the same width and height.

The dataset has color, therefore 3 channels.

The dataset is relatively big, for the porpuse of rapid iteration a subset of it is going to be selected.

# Transform Data

## Create a sub sample

In [ ]:
sub_ds_train = ds_train.take(1000)
sub_ds_test = ds_test.take(1000)
sub_ds_val = ds_val.take(1000)

In [ ]:
count_n_images_iterator(sub_ds_train, 'train'),
count_n_images_iterator(sub_ds_test, 'test'),
count_n_images_iterator(sub_ds_val, 'validation')

## Normalize data

In [ ]:
def normalize_image(image, label):
    return tf.cast(image, tf.float32) /255.0, label

In [ ]:

normalize_ds_train = sub_ds_train.map(normalize_image)
normalize_ds_test = sub_ds_test.map(normalize_image)
normalize_ds_val = sub_ds_val.map(normalize_image)

## Resize images

In [ ]:
def resize_image(image, label, target_height=100, target_width=100):
    return tf.image.resize_with_pad(image, target_height, target_width), label

In [ ]:
resized_ds_train = normalize_ds_train.map(resize_image)
resized_ds_test = normalize_ds_test.map(resize_image)
resized_ds_val = normalize_ds_val.map(resize_image)

In [ ]:
count_n_images_iterator(resized_ds_train, 'train'),
count_n_images_iterator(resized_ds_test, 'test'),
count_n_images_iterator(resized_ds_val, 'validation')

## Data after transformation

In [ ]:
fig = tfds.show_examples(resized_ds_train, ds_info, rows=4, cols=4)

## Define batch size

In [ ]:
#resized_ds_train = resized_ds_train.batch(256) # depende da memoria disponiveis na maquina, numero maior representa mais dados em memoria
#resized_ds_test = resized_ds_test.batch(256) # depende da memoria disponiveis na maquina, numero maior representa mais dados em memoria
#resized_ds_val = resized_ds_val.batch(256) # depende da memoria disponiveis na maquina, numero maior representa mais dados em memoria

resized_ds_train = resized_ds_train.batch(128) # depende da memoria disponiveis na maquina, numero maior representa mais dados em memoria
resized_ds_test = resized_ds_test.batch(128) # depende da memoria disponiveis na maquina, numero maior representa mais dados em memoria
resized_ds_val = resized_ds_val.batch(128) # depende da memoria disponiveis na maquina, numero maior representa mais dados em memoria

In [ ]:
count_n_images_iterator(resized_ds_train, 'train', 'batches'), # nao sao imagens, sao batches (grupos) de imagens
count_n_images_iterator(resized_ds_test, 'test', 'batches'), # nao sao imagens, sao batches (grupos) de imagens
count_n_images_iterator(resized_ds_val, 'validation', 'batches') # nao sao imagens, sao batches (grupos) de imagens

# Problem definition

## Objective

## Metric

Currently using 'accuracy'.

# Modeling

## Baseline model

In [ ]:
baseline_model = Sequential([
  layers.Input(shape=(100, 100, 3)),
  layers.Flatten(),
  layers.Dense(300, activation='relu'),
  layers.Dense(100, activation='relu'),
  layers.Dense(10, activation='relu'),
  layers.Dense(1, activation='linear')
])

In [ ]:
baseline_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy'])

In [ ]:
baseline_model.summary()

In [ ]:
epochs=30
history = baseline_model.fit(
  resized_ds_train,
  validation_data=resized_ds_val,
  epochs=epochs
)

In [ ]:
plot_acuracy_loss(history)

Discuss the model performance.

## CNN model

In [ ]:
cnn_model = Sequential([
  layers.Input(shape=(100, 100, 3)),
  layers.Conv2D(16, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(32, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dense(64, activation='relu'),
  layers.Dense(32, activation='relu'),
  layers.Dense(1, activation='linear')
])

In [ ]:
cnn_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy'])

In [ ]:
cnn_model.summary()

In [ ]:
%time
epochs=20
history = cnn_model.fit(
  resized_ds_train,
  validation_data=resized_ds_val,
  epochs=epochs
)

In [ ]:
plot_acuracy_loss(history)

Discuss the model performance.

# Dif CNN

In [ ]:
cnn_model2 = Sequential([
  layers.Input(shape=(100, 100, 3)),
  layers.Conv2D(16, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(32, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(128, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dense(64, activation='relu'),
  layers.Dense(32, activation='relu'),
  layers.Dense(1, activation='linear')
])

In [ ]:
cnn_model2.compile(
    optimizer='adam',
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy'])

In [ ]:
cnn_model2.summary()

In [ ]:
%time
epochs=20
history2 = cnn_model2.fit(
  resized_ds_train,
  validation_data=resized_ds_val,
  epochs=epochs
)

In [ ]:
plot_acuracy_loss(history2)

## Regularization with dropout

In [ ]:
drop_out_model = Sequential([
  layers.Input(shape=(100, 100, 3)),
  layers.Conv2D(16, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Dropout(0.01),
  layers.Conv2D(32, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Dropout(0.01),
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dropout(0.01),
  layers.Dense(64, activation='relu'),
  layers.Dropout(0.01),
  layers.Dense(32, activation='relu'),
  layers.Dropout(0.01),
  layers.Dense(1, activation='linear')
])

In [ ]:
drop_out_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy'])

In [ ]:
drop_out_model.summary()

In [ ]:
epochs=20
history = drop_out_model.fit(
  resized_ds_train,
  validation_data=resized_ds_val,
  epochs=epochs
)

In [ ]:
plot_acuracy_loss(history)

Discuss the model performance.

## Add L2 Regularization
On top of the dropout

In [ ]:
kernel_regularizer = tf.keras.regularizers.L2(0.01)

In [ ]:
l2_model = Sequential([
  layers.Input(shape=(100, 100, 3)),
  layers.Conv2D(16, 3, padding='same', activation='relu', kernel_regularizer=kernel_regularizer),
  layers.MaxPooling2D(),
  layers.Dropout(0.01),
  layers.Conv2D(32, 3, padding='same', activation='relu', kernel_regularizer=kernel_regularizer),
  layers.MaxPooling2D(),
  layers.Dropout(0.01),
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dropout(0.01),
  layers.Dense(64, activation='relu'),
  layers.Dropout(0.01),
  layers.Dense(32, activation='relu'),
  layers.Dropout(0.01),
  layers.Dense(1, activation='linear')
])

In [ ]:
l2_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy'])

In [ ]:
l2_model.summary()

In [ ]:
epochs=20
history = l2_model.fit(
  resized_ds_train,
  validation_data=resized_ds_val,
  epochs=epochs
)

In [ ]:
plot_acuracy_loss(history)

Discuss the model performance.

# The utility of the model (s) created

## Predict for the test set

# Future work

In [ ]:
kernel_regularizer = tf.keras.regularizers.L2(0.01)

In [ ]:
cnn_model3 = Sequential([
  layers.Input(shape=(100, 100, 3)),
  layers.Conv2D(16, 3, padding='same', activation='relu', kernel_regularizer=kernel_regularizer),
  layers.MaxPooling2D(),
  layers.Conv2D(32, 3, padding='valid', activation='relu', kernel_regularizer=kernel_regularizer),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, padding='valid', activation='relu', kernel_regularizer=kernel_regularizer),
  layers.MaxPooling2D(),
  layers.Conv2D(128, 3, padding='valid', activation='relu', kernel_regularizer=kernel_regularizer),
  layers.MaxPooling2D(),
  layers.Flatten(),
  layers.Dense(128, activation='relu', kernel_regularizer=kernel_regularizer),
  layers.Dense(64, activation='relu', kernel_regularizer=kernel_regularizer),
  layers.Dense(32, activation='relu', kernel_regularizer=kernel_regularizer),
  layers.Dense(1, activation='linear')
])

In [ ]:
cnn_model3.compile(
    optimizer='adam',
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy'])

In [ ]:
cnn_model3.summary()

In [ ]:
epochs=30
history3 = cnn_model3.fit(
  resized_ds_train,
  validation_data=resized_ds_val,
  epochs=epochs
)

In [ ]:
plot_acuracy_loss(history3)